# drift on colab

runs the whole measurement end to end: every knob swept over the cifar-10 test set and the ag news test set, activations hooked at every layer, analysis, charts, summary.

Runtime > Change runtime type > **T4 GPU** for the gpu sweep, or a cpu runtime for the cpu-only sweep, then Runtime > **Run all**. the last cells print the results folder (minus the regenerable parts) as base64 text and serve it as a zip.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
import torch, torchvision; print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'cuda', torch.version.cuda, 'cudnn', torch.backends.cudnn.version())

In [ ]:
import os
if not os.path.exists('/content/drift'):
    !git clone --quiet https://github.com/noorps/drift.git /content/drift
%cd /content/drift
!git pull --quiet; git log --oneline -1

In [ ]:
import time, os, torch
prefix = torch.cuda.get_device_name(0).split()[-1].lower() if torch.cuda.is_available() else 'cpu'
LABEL = os.environ.get('DRIFT_LABEL') or time.strftime(prefix + '-%Y%m%d-%H%M')
os.makedirs(f'results/{LABEL}', exist_ok=True)
print(LABEL)
!bash run.sh $LABEL 2>&1 | tee results/$LABEL/run.log

In [ ]:
from IPython.display import Image, display
for f in ('depth.png', 'flips_by_margin.png', 'knob_cost.png', 'depth_text.png', 'flips_by_margin_text.png', 'knob_cost_text.png'):
    p = f'results/{LABEL}/{f}'
    if os.path.exists(p):
        display(Image(p, width=1000))
    else:
        print('missing', p)

In [ ]:
print(open(f'results/{LABEL}/summary.md').read())

In [ ]:
# pack the json (not the logits, flip images or charts: those are regenerated from it) into base64 text,
# printed here in 20k-character pieces, and also served with a full zip through colab's port proxy.
import shutil, threading, http.server, functools
!python -m drift.pack pack results/$LABEL
shutil.make_archive(f'/content/{LABEL}', 'zip', '/content/drift/results', LABEL)
shutil.copytree(f'results/{LABEL}.b64', f'/content/{LABEL}.b64', dirs_exist_ok=True)
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory='/content')
try:
    threading.Thread(target=http.server.ThreadingHTTPServer(('127.0.0.1', 8000), handler).serve_forever, daemon=True).start()
except OSError:
    pass
from google.colab import output
base = output.eval_js('google.colab.kernel.proxyPort(8000)')
chunks = sorted(f for f in os.listdir(f'/content/{LABEL}.b64') if f.startswith('chunk_'))
print('zip:', base + f'/{LABEL}.zip')
print('sha256:', open(f'/content/{LABEL}.b64/sha256.txt').read())
print('chunks:', len(chunks))
for c in chunks:
    print(base + f'/{LABEL}.b64/{c}')

In [ ]:
# the packed results as one base64 string in this cell's output, so they survive a runtime disconnect
b = ''.join(open(f'/content/{LABEL}.b64/{c}').read().strip() for c in chunks)
print('B64LEN', len(b), 'SHA', open(f'/content/{LABEL}.b64/sha256.txt').read().strip())
print('B64BEGIN')
print(b)
print('B64END')